# Álgebra Lineal para Machine Learning y Deep Learning

## Notebook educativo — Implementación desde cero + NumPy

### Objetivo
Dominar los conceptos fundamentales de álgebra lineal que son **esenciales** para entender y desarrollar algoritmos de Machine Learning y Deep Learning.

### Contenido
1. **Vectores**: operaciones, producto punto, normas, proyecciones
2. **Matrices**: operaciones, transpuesta, inversa, determinante
3. **Sistemas de ecuaciones lineales**: eliminación gaussiana, mínimos cuadrados
4. **Espacios vectoriales**: independencia lineal, base, rango, nulidad
5. **Transformaciones lineales**: rotación, escala, interpretación geométrica
6. **Valores y vectores propios (Eigendecomposition)**: diagonalización, conexión con PCA
7. **Descomposiciones matriciales**: SVD, LU, QR, Cholesky
8. **Matrices especiales**: simétrica, ortogonal, definida positiva, sparse
9. **Tensores**: operaciones para Deep Learning
10. **Aplicaciones en ML/DL**: covarianza, gradientes, Hessiano, batch operations

### Requisitos
```
numpy matplotlib
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
np.random.seed(42)
np.set_printoptions(precision=4, suppress=True)
print("✅ Entorno listo")

## 1. Vectores — El bloque fundamental

### ¿Qué es un vector?
Un vector es una **lista ordenada de números** que representa un punto o dirección en un espacio n-dimensional.

$$\vec{v} = \begin{bmatrix} v_1 \\ v_2 \\ \vdots \\ v_n \end{bmatrix} \in \mathbb{R}^n$$

### En ML, los vectores representan:
- **Features** de un ejemplo: $\vec{x} = [edad, ingreso, score]$
- **Pesos** de un modelo: $\vec{w} = [w_1, w_2, ..., w_n]$
- **Embeddings**: representaciones densas de palabras, imágenes, etc.
- **Gradientes**: dirección de máximo cambio de la función de pérdida

### Operaciones fundamentales

| Operación | Fórmula | Uso en ML |
|-----------|---------|-----------|
| Suma | $\vec{u} + \vec{v} = [u_i + v_i]$ | Actualización de pesos |
| Escalar | $c\vec{v} = [cv_i]$ | Learning rate × gradiente |
| Producto punto | $\vec{u} \cdot \vec{v} = \sum u_i v_i$ | Neurona, similitud |
| Norma L2 | $\|\vec{v}\| = \sqrt{\sum v_i^2}$ | Regularización |
| Norma L1 | $\|\vec{v}\|_1 = \sum |v_i|$ | Regularización Lasso |

In [ ]:
# ============================================================
# VECTORES — Implementación desde cero
# ============================================================

class Vector:
    """Implementación de vector desde cero."""
    
    def __init__(self, componentes):
        self.data = list(componentes)
        self.n = len(self.data)
    
    def __repr__(self):
        return f"Vector({self.data})"
    
    def __add__(self, otro):
        assert self.n == otro.n, "Dimensiones incompatibles"
        return Vector([a + b for a, b in zip(self.data, otro.data)])
    
    def __sub__(self, otro):
        assert self.n == otro.n, "Dimensiones incompatibles"
        return Vector([a - b for a, b in zip(self.data, otro.data)])
    
    def __mul__(self, escalar):
        return Vector([escalar * x for x in self.data])
    
    def __rmul__(self, escalar):
        return self.__mul__(escalar)
    
    def producto_punto(self, otro):
        """Producto punto (dot product): u·v = Σ(ui * vi)"""
        assert self.n == otro.n, "Dimensiones incompatibles"
        return sum(a * b for a, b in zip(self.data, otro.data))
    
    def norma_l2(self):
        """Norma euclidiana: ||v|| = sqrt(Σ vi²)"""
        return sum(x**2 for x in self.data) ** 0.5
    
    def norma_l1(self):
        """Norma Manhattan: ||v||₁ = Σ|vi|"""
        return sum(abs(x) for x in self.data)
    
    def norma_linf(self):
        """Norma infinito: ||v||∞ = max|vi|"""
        return max(abs(x) for x in self.data)
    
    def normalizar(self):
        """Vector unitario: v/||v||"""
        norma = self.norma_l2()
        if norma == 0:
            return Vector([0] * self.n)
        return Vector([x / norma for x in self.data])
    
    def proyeccion_sobre(self, otro):
        """Proyección de self sobre otro: proj_b(a) = (a·b/b·b) * b"""
        escalar = self.producto_punto(otro) / otro.producto_punto(otro)
        return escalar * otro
    
    def angulo_con(self, otro):
        """Ángulo entre dos vectores en radianes."""
        cos_theta = self.producto_punto(otro) / (self.norma_l2() * otro.norma_l2())
        cos_theta = max(-1, min(1, cos_theta))  # Clamp por errores numéricos
        return np.arccos(cos_theta)
    
    def similitud_coseno(self, otro):
        """Similitud coseno: cos(θ) = (u·v)/(||u|| ||v||)"""
        norma_prod = self.norma_l2() * otro.norma_l2()
        if norma_prod == 0:
            return 0
        return self.producto_punto(otro) / norma_prod

# --- Demostración ---
u = Vector([3, 4])
v = Vector([1, 2])

print("=== Operaciones con Vectores ===")
print(f"u = {u}")
print(f"v = {v}")
print(f"u + v = {u + v}")
print(f"u - v = {u - v}")
print(f"3 * u = {3 * u}")
print(f"u · v = {u.producto_punto(v)}")
print(f"||u||₂ = {u.norma_l2():.4f}")
print(f"||u||₁ = {u.norma_l1():.4f}")
print(f"||u||∞ = {u.norma_linf():.4f}")
print(f"û (unitario) = {u.normalizar()}")
print(f"Ángulo(u,v) = {np.degrees(u.angulo_con(v)):.2f}°")
print(f"Similitud coseno = {u.similitud_coseno(v):.4f}")
print(f"Proyección de u sobre v = {u.proyeccion_sobre(v)}")

In [ ]:
# ============================================================
# Visualización: Operaciones con vectores en 2D
# ============================================================

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

u = np.array([3, 1])
v = np.array([1, 3])

def plot_vector(ax, origin, vec, color, label, lw=2):
    ax.annotate('', xy=origin + vec, xytext=origin,
                arrowprops=dict(arrowstyle='->', color=color, lw=lw))
    ax.text(origin[0] + vec[0]/2 + 0.1, origin[1] + vec[1]/2 + 0.1, 
            label, color=color, fontsize=11, fontweight='bold')

# 1. Suma de vectores
ax = axes[0, 0]
plot_vector(ax, np.array([0,0]), u, 'blue', 'u')
plot_vector(ax, np.array([0,0]), v, 'red', 'v')
plot_vector(ax, np.array([0,0]), u+v, 'green', 'u+v', lw=2.5)
plot_vector(ax, u, v, 'red', '', lw=1)  # Regla del paralelogramo
ax.set_xlim(-0.5, 5); ax.set_ylim(-0.5, 5); ax.grid(True, alpha=0.3)
ax.set_title('Suma de vectores', fontweight='bold'); ax.set_aspect('equal')

# 2. Producto escalar
ax = axes[0, 1]
plot_vector(ax, np.array([0,0]), u, 'blue', 'u')
plot_vector(ax, np.array([0,0]), 2*u, 'purple', '2u', lw=2.5)
plot_vector(ax, np.array([0,0]), -1*u, 'orange', '-u')
ax.set_xlim(-4, 7); ax.set_ylim(-2, 4); ax.grid(True, alpha=0.3)
ax.set_title('Multiplicación escalar', fontweight='bold'); ax.set_aspect('equal')

# 3. Producto punto (proyección)
ax = axes[0, 2]
plot_vector(ax, np.array([0,0]), u, 'blue', 'u')
plot_vector(ax, np.array([0,0]), v, 'red', 'v')
proj = (np.dot(u, v) / np.dot(v, v)) * v
plot_vector(ax, np.array([0,0]), proj, 'green', 'proj_v(u)', lw=2.5)
ax.plot([u[0], proj[0]], [u[1], proj[1]], 'k--', alpha=0.5)
ax.set_xlim(-0.5, 4); ax.set_ylim(-0.5, 4); ax.grid(True, alpha=0.3)
ax.set_title('Proyección de u sobre v', fontweight='bold'); ax.set_aspect('equal')

# 4. Normas (bolas unitarias)
ax = axes[1, 0]
theta = np.linspace(0, 2*np.pi, 200)
# L2
ax.plot(np.cos(theta), np.sin(theta), 'b-', lw=2, label='L2 (círculo)')
# L1
t = np.linspace(0, 1, 50)
l1_x = np.concatenate([t, -t, -t, t])
l1_y = np.concatenate([1-t, t-0, -(1-t), -(t-0)])
ax.plot(l1_x, l1_y, 'r-', lw=2, label='L1 (diamante)')
# L∞
ax.plot([-1,1,1,-1,-1], [-1,-1,1,1,-1], 'g-', lw=2, label='L∞ (cuadrado)')
ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5); ax.grid(True, alpha=0.3)
ax.legend(); ax.set_title('Bolas unitarias (normas)', fontweight='bold'); ax.set_aspect('equal')

# 5. Similitud coseno
ax = axes[1, 1]
angulos = [0, 30, 60, 90, 120, 180]
colores = plt.cm.RdYlGn_r(np.linspace(0, 1, len(angulos)))
for ang, color in zip(angulos, colores):
    rad = np.radians(ang)
    vx, vy = np.cos(rad), np.sin(rad)
    cos_sim = np.cos(rad)
    plot_vector(ax, np.array([0,0]), np.array([vx, vy])*2, color, 
                f'{ang}° (cos={cos_sim:.2f})')
ax.set_xlim(-2.5, 2.5); ax.set_ylim(-0.5, 2.5); ax.grid(True, alpha=0.3)
ax.set_title('Similitud coseno', fontweight='bold'); ax.set_aspect('equal')

# 6. Ejemplo ML: neurona como producto punto
ax = axes[1, 2]
np.random.seed(42)
X = np.random.randn(50, 2)
w = np.array([1.5, -1])
y = (X @ w > 0).astype(int)
colors = ['red' if yi == 0 else 'blue' for yi in y]
ax.scatter(X[:, 0], X[:, 1], c=colors, alpha=0.6, s=50)
# Frontera de decisión: w·x = 0
x_line = np.linspace(-3, 3, 100)
ax.plot(x_line, w[0]/w[1] * x_line, 'k-', lw=2, label='w·x = 0')
plot_vector(ax, np.array([0,0]), w/np.linalg.norm(w)*1.5, 'green', 'w')
ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.grid(True, alpha=0.3)
ax.set_title('Neurona: w·x > 0 → clase', fontweight='bold')
ax.legend(); ax.set_aspect('equal')

plt.tight_layout()
plt.savefig('vectores_operaciones.png', dpi=100, bbox_inches='tight')
plt.show()
print("✅ Visualización de vectores completada")

## 2. Matrices — Transformaciones y datos

### ¿Qué es una matriz?
Una matriz $A \in \mathbb{R}^{m \times n}$ es un arreglo rectangular de números con $m$ filas y $n$ columnas.

$$A = \begin{bmatrix} a_{11} & a_{12} & \cdots & a_{1n} \\ a_{21} & a_{22} & \cdots & a_{2n} \\ \vdots & \vdots & \ddots & \vdots \\ a_{m1} & a_{m2} & \cdots & a_{mn} \end{bmatrix}$$

### En ML, las matrices representan:
| Concepto | Dimensión | Descripción |
|----------|-----------|-------------|
| Dataset | $(n_{samples}, n_{features})$ | Cada fila es un ejemplo |
| Pesos de capa | $(n_{in}, n_{out})$ | Transformación lineal |
| Covarianza | $(n, n)$ | Relaciones entre features |
| Kernel/Filtro | $(k, k)$ | Detección de patrones (CNN) |
| Atención | $(seq, seq)$ | Relaciones entre tokens |

### Operaciones clave

- **Multiplicación**: $C = AB$ donde $c_{ij} = \sum_k a_{ik} b_{kj}$ — **la operación más importante en DL**
- **Transpuesta**: $(A^T)_{ij} = A_{ji}$ — cambiar filas por columnas
- **Inversa**: $A^{-1}A = I$ — "deshacer" una transformación
- **Determinante**: $\det(A)$ — factor de escala de la transformación
- **Traza**: $\text{tr}(A) = \sum_i a_{ii}$ — suma de la diagonal

In [ ]:
# ============================================================
# MATRICES — Implementación desde cero
# ============================================================

class Matriz:
    """Implementación de matriz desde cero."""
    
    def __init__(self, datos):
        if isinstance(datos[0], (list, tuple)):
            self.data = [list(fila) for fila in datos]
        else:
            self.data = [list(datos)]
        self.filas = len(self.data)
        self.cols = len(self.data[0])
    
    def __repr__(self):
        filas_str = '\n  '.join(str(f) for f in self.data)
        return f"Matriz({self.filas}x{self.cols}):\n  {filas_str}"
    
    def __getitem__(self, idx):
        i, j = idx
        return self.data[i][j]
    
    def __setitem__(self, idx, val):
        i, j = idx
        self.data[i][j] = val
    
    def __add__(self, otra):
        assert (self.filas, self.cols) == (otra.filas, otra.cols)
        return Matriz([[self[i,j] + otra[i,j] for j in range(self.cols)] 
                       for i in range(self.filas)])
    
    def __mul__(self, escalar):
        return Matriz([[self[i,j] * escalar for j in range(self.cols)] 
                       for i in range(self.filas)])
    
    def multiplicar(self, otra):
        """Multiplicación de matrices: C = A @ B."""
        assert self.cols == otra.filas, f"Dimensiones incompatibles: {self.cols} != {otra.filas}"
        resultado = [[0] * otra.cols for _ in range(self.filas)]
        for i in range(self.filas):
            for j in range(otra.cols):
                for k in range(self.cols):
                    resultado[i][j] += self[i,k] * otra[k,j]
        return Matriz(resultado)
    
    def transpuesta(self):
        """A^T: intercambia filas y columnas."""
        return Matriz([[self[j,i] for j in range(self.filas)] 
                       for i in range(self.cols)])
    
    def traza(self):
        """tr(A) = Σ a_ii"""
        assert self.filas == self.cols, "La traza requiere matriz cuadrada"
        return sum(self[i,i] for i in range(self.filas))
    
    def determinante(self):
        """Determinante por expansión de cofactores (recursivo)."""
        assert self.filas == self.cols
        n = self.filas
        if n == 1:
            return self[0,0]
        if n == 2:
            return self[0,0]*self[1,1] - self[0,1]*self[1,0]
        det = 0
        for j in range(n):
            submatriz = Matriz([[self[i,k] for k in range(n) if k != j] 
                               for i in range(1, n)])
            det += ((-1)**j) * self[0,j] * submatriz.determinante()
        return det
    
    def inversa(self):
        """Inversa por método de Gauss-Jordan."""
        assert self.filas == self.cols
        n = self.filas
        # Matriz aumentada [A | I]
        aug = [[self[i,j] for j in range(n)] + [1.0 if i==k else 0.0 for k in range(n)] 
               for i in range(n)]
        
        for col in range(n):
            # Pivoteo parcial
            max_row = max(range(col, n), key=lambda r: abs(aug[r][col]))
            aug[col], aug[max_row] = aug[max_row], aug[col]
            
            pivot = aug[col][col]
            assert abs(pivot) > 1e-10, "Matriz singular"
            aug[col] = [x / pivot for x in aug[col]]
            
            for row in range(n):
                if row != col:
                    factor = aug[row][col]
                    aug[row] = [aug[row][k] - factor * aug[col][k] for k in range(2*n)]
        
        return Matriz([[aug[i][j+n] for j in range(n)] for i in range(n)])
    
    @staticmethod
    def identidad(n):
        return Matriz([[1.0 if i==j else 0.0 for j in range(n)] for i in range(n)])

# --- Demostración ---
A = Matriz([[1, 2], [3, 4]])
B = Matriz([[5, 6], [7, 8]])

print("=== Operaciones con Matrices ===")
print(f"A =\n{A}\n")
print(f"B =\n{B}\n")
print(f"A + B =\n{A + B}\n")
print(f"A × B =\n{A.multiplicar(B)}\n")
print(f"A^T =\n{A.transpuesta()}\n")
print(f"tr(A) = {A.traza()}")
print(f"det(A) = {A.determinante()}")
print(f"\nA^(-1) =\n{A.inversa()}")
print(f"\nVerificación A × A^(-1) =\n{A.multiplicar(A.inversa())}")

In [ ]:
# ============================================================
# Visualización: Matrices como transformaciones lineales
# ============================================================

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Puntos del cuadrado unitario
theta = np.linspace(0, 2*np.pi, 100)
circle = np.column_stack([np.cos(theta), np.sin(theta)])
square = np.array([[0,0],[1,0],[1,1],[0,1],[0,0]])

transformaciones = {
    'Identidad': np.eye(2),
    'Escala (2x, 0.5y)': np.array([[2, 0], [0, 0.5]]),
    'Rotación 45°': np.array([[np.cos(np.pi/4), -np.sin(np.pi/4)],
                               [np.sin(np.pi/4), np.cos(np.pi/4)]]),
    'Shear': np.array([[1, 0.5], [0, 1]]),
    'Reflexión (eje x)': np.array([[1, 0], [0, -1]]),
    'Singular (det=0)': np.array([[1, 2], [0.5, 1]])
}

for ax, (nombre, M) in zip(axes.flat, transformaciones.items()):
    # Transformar
    circle_t = circle @ M.T
    square_t = square @ M.T
    
    ax.plot(circle[:, 0], circle[:, 1], 'b-', alpha=0.3, lw=1, label='Original')
    ax.plot(circle_t[:, 0], circle_t[:, 1], 'r-', lw=2, label='Transformado')
    ax.plot(square[:, 0], square[:, 1], 'b--', alpha=0.3, lw=1)
    ax.plot(square_t[:, 0], square_t[:, 1], 'r--', lw=2)
    
    det = np.linalg.det(M)
    ax.set_title(f'{nombre}\ndet = {det:.2f}', fontweight='bold')
    ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
    ax.grid(True, alpha=0.3); ax.set_aspect('equal')
    ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('transformaciones_lineales.png', dpi=100, bbox_inches='tight')
plt.show()
print("✅ El determinante indica cuánto se escala el área. Si det=0, la transformación colapsa dimensiones.")

## 3. Sistemas de Ecuaciones Lineales

### Forma matricial
Un sistema $Ax = b$ donde:
- $A \in \mathbb{R}^{m \times n}$ es la matriz de coeficientes
- $x \in \mathbb{R}^n$ es el vector de incógnitas  
- $b \in \mathbb{R}^m$ es el vector de resultados

### Métodos de solución

| Método | Complejidad | Uso |
|--------|-------------|-----|
| Eliminación Gaussiana | $O(n^3)$ | Solución exacta (sistema cuadrado) |
| Mínimos cuadrados | $x = (A^TA)^{-1}A^Tb$ | Sobredeterminado (regresión lineal!) |
| Pseudoinversa | $x = A^+b$ | Caso general |

### Conexión con ML
La **regresión lineal** es resolver el sistema sobredeterminado $X\vec{w} = \vec{y}$, y la solución analítica es exactamente:

$$\vec{w}^* = (X^TX)^{-1}X^T\vec{y}$$

Esta es la **ecuación normal** — ¡puro álgebra lineal!

In [ ]:
# ============================================================
# Eliminación Gaussiana desde cero
# ============================================================

def eliminacion_gaussiana(A, b):
    """Resuelve Ax = b por eliminación gaussiana con pivoteo parcial."""
    n = len(b)
    # Crear matriz aumentada
    M = [list(A[i]) + [b[i]] for i in range(n)]
    
    # Eliminación hacia adelante
    for col in range(n):
        # Pivoteo parcial
        max_row = max(range(col, n), key=lambda r: abs(M[r][col]))
        M[col], M[max_row] = M[max_row], M[col]
        
        for row in range(col + 1, n):
            factor = M[row][col] / M[col][col]
            for j in range(col, n + 1):
                M[row][j] -= factor * M[col][j]
    
    # Sustitución hacia atrás
    x = [0.0] * n
    for i in range(n - 1, -1, -1):
        x[i] = (M[i][n] - sum(M[i][j] * x[j] for j in range(i+1, n))) / M[i][i]
    
    return x

def minimos_cuadrados(X, y):
    """Solución de mínimos cuadrados: w = (X^T X)^(-1) X^T y."""
    XtX = X.T @ X
    Xty = X.T @ y
    # Resolvemos usando numpy por estabilidad
    w = np.linalg.solve(XtX, Xty)
    return w

# --- Ejemplo 1: Sistema exacto ---
print("=== Sistema Exacto: 3 ecuaciones, 3 incógnitas ===")
print("2x + y - z = 8")
print("-3x - y + 2z = -11")
print("-2x + y + 2z = -3")

A = [[2, 1, -1], [-3, -1, 2], [-2, 1, 2]]
b = [8, -11, -3]
x = eliminacion_gaussiana(A, b)
print(f"Solución: x={x[0]:.4f}, y={x[1]:.4f}, z={x[2]:.4f}")

# Verificar con NumPy
x_np = np.linalg.solve(np.array(A), np.array(b))
print(f"NumPy:    x={x_np[0]:.4f}, y={x_np[1]:.4f}, z={x_np[2]:.4f}")

# --- Ejemplo 2: Regresión lineal como mínimos cuadrados ---
print("\n=== Regresión Lineal = Mínimos Cuadrados ===")
np.random.seed(42)
n_puntos = 100
x_datos = np.random.uniform(0, 10, n_puntos)
y_datos = 3 * x_datos + 7 + np.random.randn(n_puntos) * 2  # y = 3x + 7 + ruido

# Agregar columna de 1s (bias)
X = np.column_stack([x_datos, np.ones(n_puntos)])
w = minimos_cuadrados(X, y_datos)
print(f"Pesos encontrados: pendiente={w[0]:.4f}, intercepto={w[1]:.4f}")
print(f"Valores reales:    pendiente=3.0000, intercepto=7.0000")

# Visualizar
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x_datos, y_datos, alpha=0.5, s=20, label='Datos')
x_line = np.linspace(0, 10, 100)
ax.plot(x_line, w[0]*x_line + w[1], 'r-', lw=2, 
        label=f'y = {w[0]:.2f}x + {w[1]:.2f}')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('Regresión Lineal = Resolver Ax = b por Mínimos Cuadrados', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 4. Espacios Vectoriales, Rango y Nulidad

### Conceptos clave

**Independencia lineal**: Los vectores $\{v_1, ..., v_k\}$ son linealmente independientes si:
$$c_1 v_1 + c_2 v_2 + ... + c_k v_k = 0 \implies c_1 = c_2 = ... = c_k = 0$$

**Rango (rank)**: Número de filas/columnas linealmente independientes.
$$\text{rank}(A) = \dim(\text{espacio columna de } A)$$

**Nulidad**: Dimensión del espacio nulo (soluciones de $Ax = 0$).
$$\text{rank}(A) + \text{nulidad}(A) = n \text{ (número de columnas)}$$

### En ML:
- **Rango de la matriz de datos** $X$: indica cuántas features son realmente independientes
- Si $\text{rank}(X) < n_{features}$: hay **multicolinealidad** → problemas en regresión
- **PCA** encuentra las direcciones de mayor varianza en el espacio columna

In [ ]:
# ============================================================
# Rango, independencia lineal y nulidad desde cero
# ============================================================

def forma_escalonada(A):
    """Reduce A a forma escalonada por filas (REF)."""
    M = [list(fila) for fila in A]
    filas, cols = len(M), len(M[0])
    pivote_fila = 0
    
    for col in range(cols):
        if pivote_fila >= filas:
            break
        # Buscar pivote
        max_row = max(range(pivote_fila, filas), key=lambda r: abs(M[r][col]))
        if abs(M[max_row][col]) < 1e-10:
            continue
        M[pivote_fila], M[max_row] = M[max_row], M[pivote_fila]
        
        # Eliminar debajo
        for row in range(pivote_fila + 1, filas):
            factor = M[row][col] / M[pivote_fila][col]
            for j in range(cols):
                M[row][j] -= factor * M[pivote_fila][j]
        pivote_fila += 1
    
    return M

def rango(A):
    """Calcula el rango contando filas no nulas en la forma escalonada."""
    ref = forma_escalonada(A)
    return sum(1 for fila in ref if any(abs(x) > 1e-10 for x in fila))

# --- Demostración ---
print("=== Rango e Independencia Lineal ===\n")

# Caso 1: Rango completo
A1 = [[1, 0, 0], [0, 1, 0], [0, 0, 1]]
print(f"Identidad 3x3: rango = {rango(A1)} (completo)")

# Caso 2: Columnas dependientes
A2 = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
print(f"Filas en progresión: rango = {rango(A2)} (no completo, fila 3 = 2*fila2 - fila1)")

# Caso 3: Matriz con redundancia
A3 = [[1, 2], [2, 4], [3, 6]]
print(f"Columnas proporcionales: rango = {rango(A3)} (columna 2 = 2 * columna 1)")

# Verificar con NumPy
for nombre, mat in [("A1", A1), ("A2", A2), ("A3", A3)]:
    print(f"  NumPy rank({nombre}) = {np.linalg.matrix_rank(np.array(mat, dtype=float))}")

# --- Multicolinealidad en regresión ---
print("\n=== Multicolinealidad: Problema en ML ===")
np.random.seed(42)
n = 100
x1 = np.random.randn(n)
x2 = 2 * x1 + np.random.randn(n) * 0.01  # Casi perfectamente correlacionada
x3 = np.random.randn(n)

X_colineal = np.column_stack([x1, x2, x3, np.ones(n)])
print(f"Rango de X (4 columnas): {np.linalg.matrix_rank(X_colineal)}")
print(f"Número de condición: {np.linalg.cond(X_colineal):.1f}")
print("→ Número de condición alto = matriz mal condicionada = problemas numéricos")

X_buena = np.column_stack([x1, x3, np.random.randn(n), np.ones(n)])
print(f"\nRango de X buena: {np.linalg.matrix_rank(X_buena)}")
print(f"Número de condición: {np.linalg.cond(X_buena):.1f}")
print("→ Número de condición bajo = estable")

## 5. Valores y Vectores Propios (Eigenvalues & Eigenvectors)

### Definición
Un vector propio $\vec{v}$ de una matriz $A$ es un vector que **no cambia de dirección** al ser transformado, solo se escala:

$$A\vec{v} = \lambda \vec{v}$$

donde $\lambda$ es el **valor propio** (eigenvalue) asociado.

### Cálculo
Los valores propios son las raíces del **polinomio característico**:
$$\det(A - \lambda I) = 0$$

### ¿Por qué son cruciales en ML?

| Aplicación | Uso de eigenvalues/eigenvectors |
|-----------|-------------------------------|
| **PCA** | Los eigenvectors de la covarianza son las componentes principales |
| **Espectral clustering** | Eigenvectors del Laplaciano del grafo |
| **PageRank** | Eigenvector dominante de la matriz de transición |
| **Estabilidad** | Los eigenvalues determinan convergencia del entrenamiento |
| **Hessiano** | Eigenvalues del Hessiano → curvatura de la loss |
| **SVD** | Basado en eigendecomposition de $A^TA$ y $AA^T$ |

### Diagonalización
Si $A$ tiene $n$ eigenvectors independientes:
$$A = V \Lambda V^{-1}$$
donde $V$ = matriz de eigenvectors, $\Lambda$ = diagonal de eigenvalues.

Esto permite calcular potencias eficientemente: $A^k = V \Lambda^k V^{-1}$

In [ ]:
# ============================================================
# Eigenvalues & Eigenvectors desde cero
# ============================================================

def eigen_2x2(A):
    """Calcula eigenvalues/eigenvectors para matrices 2x2 analíticamente.
    
    Para A = [[a, b], [c, d]]:
    Polinomio característico: λ² - (a+d)λ + (ad-bc) = 0
    """
    a, b = A[0][0], A[0][1]
    c, d = A[1][0], A[1][1]
    
    # Coeficientes del polinomio característico
    traza = a + d
    det = a * d - b * c
    
    # Fórmula cuadrática
    discriminante = traza**2 - 4 * det
    
    if discriminante >= 0:
        sqrt_disc = discriminante ** 0.5
        lambda1 = (traza + sqrt_disc) / 2
        lambda2 = (traza - sqrt_disc) / 2
    else:
        # Eigenvalues complejos
        real = traza / 2
        imag = (-discriminante) ** 0.5 / 2
        return (complex(real, imag), complex(real, -imag)), None
    
    # Encontrar eigenvectors
    eigenvectors = []
    for lam in [lambda1, lambda2]:
        # (A - λI)v = 0
        if abs(b) > 1e-10:
            v = [b, lam - a]
        elif abs(c) > 1e-10:
            v = [lam - d, c]
        else:
            v = [1, 0] if abs(a - lam) < 1e-10 else [0, 1]
        # Normalizar
        norma = (v[0]**2 + v[1]**2) ** 0.5
        eigenvectors.append([v[0]/norma, v[1]/norma])
    
    return (lambda1, lambda2), eigenvectors

def power_iteration(A, num_iter=1000, tol=1e-10):
    """Método de la potencia: encuentra el eigenvalue dominante.
    
    Usado en PageRank y PCA para matrices grandes.
    """
    n = A.shape[0]
    v = np.random.randn(n)
    v = v / np.linalg.norm(v)
    
    for _ in range(num_iter):
        v_new = A @ v
        eigenvalue = np.dot(v_new, v)
        v_new = v_new / np.linalg.norm(v_new)
        if np.linalg.norm(v_new - v) < tol:
            break
        v = v_new
    
    return eigenvalue, v_new

# --- Ejemplo analítico 2x2 ---
print("=== Eigenvalues 2x2 (analítico) ===")
A = [[4, 2], [1, 3]]
eigenvalues, eigenvectors = eigen_2x2(A)
print(f"A = {A}")
print(f"Eigenvalues: λ₁={eigenvalues[0]:.4f}, λ₂={eigenvalues[1]:.4f}")
print(f"Eigenvectors: v₁={eigenvectors[0]}, v₂={eigenvectors[1]}")

# Verificar: Av = λv
A_np = np.array(A, dtype=float)
for i, (lam, v) in enumerate(zip(eigenvalues, eigenvectors)):
    v = np.array(v)
    Av = A_np @ v
    lv = lam * v
    print(f"  Av{i+1} = {Av}, λ{i+1}v{i+1} = {lv}, iguales: {np.allclose(Av, lv)}")

# --- Power Iteration ---
print("\n=== Power Iteration (eigenvalue dominante) ===")
A_grande = np.array([[5, 1, 2], [1, 4, 1], [2, 1, 3]], dtype=float)
lam_dom, v_dom = power_iteration(A_grande)
print(f"Eigenvalue dominante: {lam_dom:.6f}")
print(f"Eigenvector dominante: {v_dom}")

# Verificar con NumPy
eigenvalues_np, eigenvectors_np = np.linalg.eigh(A_grande)
print(f"NumPy eigenvalues: {eigenvalues_np}")
print(f"Eigenvalue máximo (NumPy): {eigenvalues_np[-1]:.6f}")

In [ ]:
# ============================================================
# Visualización: Eigenvectors y PCA
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Eigenvectors como direcciones invariantes
ax = axes[0]
A = np.array([[2, 1], [1, 3]])
eigenvalues, eigenvectors = np.linalg.eigh(A)

theta = np.linspace(0, 2*np.pi, 100)
circle = np.column_stack([np.cos(theta), np.sin(theta)])
ellipse = circle @ A.T

ax.plot(circle[:, 0], circle[:, 1], 'b-', alpha=0.3, lw=1, label='Círculo unitario')
ax.plot(ellipse[:, 0], ellipse[:, 1], 'r-', lw=2, label='Transformado por A')

for i, (val, vec) in enumerate(zip(eigenvalues, eigenvectors.T)):
    ax.annotate('', xy=vec*val, xytext=[0,0],
                arrowprops=dict(arrowstyle='->', color='green', lw=3))
    ax.text(vec[0]*val + 0.2, vec[1]*val + 0.2, 
            f'λ={val:.2f}', color='green', fontsize=12, fontweight='bold')

ax.set_xlim(-4, 4); ax.set_ylim(-4, 4); ax.set_aspect('equal')
ax.grid(True, alpha=0.3); ax.legend()
ax.set_title('Eigenvectors = Ejes de la elipse', fontweight='bold')

# 2. PCA: Eigenvectors de la covarianza
ax = axes[1]
np.random.seed(42)
mean = [0, 0]
cov = [[3, 2], [2, 2]]
datos = np.random.multivariate_normal(mean, cov, 200)

eigenvalues_cov, eigenvectors_cov = np.linalg.eigh(np.cov(datos.T))
# Ordenar de mayor a menor
idx = eigenvalues_cov.argsort()[::-1]
eigenvalues_cov = eigenvalues_cov[idx]
eigenvectors_cov = eigenvectors_cov[:, idx]

ax.scatter(datos[:, 0], datos[:, 1], alpha=0.3, s=10)
colores_pc = ['red', 'blue']
for i in range(2):
    vec = eigenvectors_cov[:, i] * eigenvalues_cov[i]
    ax.annotate('', xy=vec, xytext=[0,0],
                arrowprops=dict(arrowstyle='->', color=colores_pc[i], lw=3))
    ax.text(vec[0] + 0.2, vec[1] + 0.2, 
            f'PC{i+1} (λ={eigenvalues_cov[i]:.2f})', 
            color=colores_pc[i], fontsize=11, fontweight='bold')

ax.set_xlim(-5, 5); ax.set_ylim(-5, 5); ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.set_title('PCA = Eigenvectors de la Covarianza', fontweight='bold')

# 3. Varianza explicada (scree plot)
ax = axes[2]
np.random.seed(42)
X_high = np.random.randn(500, 10)
X_high[:, 1] = X_high[:, 0] * 2 + np.random.randn(500) * 0.1
X_high[:, 2] = X_high[:, 0] * 1.5 + np.random.randn(500) * 0.3
X_high[:, 3] = X_high[:, 1] + X_high[:, 2] + np.random.randn(500) * 0.2

cov_high = np.cov(X_high.T)
eigenvalues_high = np.linalg.eigvalsh(cov_high)[::-1]
varianza_explicada = eigenvalues_high / eigenvalues_high.sum() * 100
varianza_acumulada = np.cumsum(varianza_explicada)

ax.bar(range(1, 11), varianza_explicada, alpha=0.7, color='steelblue', label='Individual')
ax.plot(range(1, 11), varianza_acumulada, 'ro-', lw=2, label='Acumulada')
ax.axhline(95, color='k', linestyle='--', alpha=0.5, label='95%')
ax.set_xlabel('Componente Principal'); ax.set_ylabel('% Varianza explicada')
ax.set_title('Scree Plot: ¿Cuántas PCs conservar?', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('eigenvalues_pca.png', dpi=100, bbox_inches='tight')
plt.show()
print("✅ Los eigenvalues de la covarianza nos dicen cuánta información captura cada PC")

## 6. Descomposiciones Matriciales

### SVD (Singular Value Decomposition)
La descomposición **más importante** en ciencia de datos:

$$A = U \Sigma V^T$$

donde:
- $U \in \mathbb{R}^{m \times m}$: vectores singulares izquierdos (ortonormal)
- $\Sigma \in \mathbb{R}^{m \times n}$: valores singulares $\sigma_1 \geq \sigma_2 \geq ... \geq 0$ (diagonal)
- $V \in \mathbb{R}^{n \times n}$: vectores singulares derechos (ortonormal)

### Propiedades clave
- Funciona para **cualquier** matriz (no requiere cuadrada ni simétrica)
- $\sigma_i = \sqrt{\lambda_i(A^TA)}$
- Rango de $A$ = número de valores singulares no nulos
- **Mejor aproximación de rango bajo**: truncar a los $k$ mayores valores singulares

### Otras descomposiciones

| Descomposición | Forma | Requisito | Uso en ML |
|----------------|-------|-----------|-----------|
| **LU** | $A = LU$ | Cuadrada | Resolver sistemas |
| **QR** | $A = QR$ | Cualquiera | Mínimos cuadrados, estabilidad |
| **Cholesky** | $A = LL^T$ | Simétrica def. positiva | Gaussian Processes, sampling |
| **Eigen** | $A = V\Lambda V^{-1}$ | Cuadrada | PCA, análisis espectral |
| **SVD** | $A = U\Sigma V^T$ | Cualquiera | Reducción dimensión, NLP |

In [ ]:
# ============================================================
# SVD desde cero (via eigendecomposition de A^T A)
# ============================================================

def svd_desde_cero(A, k=None):
    """SVD: A = U Σ V^T
    
    Pasos:
    1. Calcular A^T A
    2. Eigendecomposition de A^T A → V y σ²
    3. σ = sqrt(eigenvalues)
    4. U = AV / σ
    """
    m, n = A.shape
    
    # Paso 1: A^T A (n×n)
    AtA = A.T @ A
    
    # Paso 2: Eigenvalues y eigenvectors de A^T A
    eigenvalues, V = np.linalg.eigh(AtA)
    
    # Ordenar de mayor a menor
    idx = eigenvalues.argsort()[::-1]
    eigenvalues = eigenvalues[idx]
    V = V[:, idx]
    
    # Paso 3: Valores singulares
    sigma = np.sqrt(np.maximum(eigenvalues, 0))
    
    # Paso 4: U = AV / σ
    r = np.sum(sigma > 1e-10)  # Rango
    U = np.zeros((m, r))
    for i in range(r):
        U[:, i] = (A @ V[:, i]) / sigma[i]
    
    if k is not None:
        return U[:, :k], sigma[:k], V[:, :k].T
    
    return U, sigma[:r], V[:, :r].T

# --- Ejemplo: Compresión de matriz con SVD truncada ---
print("=== SVD desde cero ===")
A = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9],
              [10, 11, 12]], dtype=float)

U, sigma, Vt = svd_desde_cero(A)
print(f"A shape: {A.shape}")
print(f"U shape: {U.shape}")
print(f"Σ (valores singulares): {sigma}")
print(f"V^T shape: {Vt.shape}")

# Reconstruir
A_reconstruida = U @ np.diag(sigma) @ Vt
print(f"\n||A - UΣV^T|| = {np.linalg.norm(A - A_reconstruida):.10f}")

# Verificar con NumPy
U_np, s_np, Vt_np = np.linalg.svd(A, full_matrices=False)
print(f"NumPy Σ: {s_np}")

# --- Aproximación de rango bajo ---
print("\n=== Aproximación de Rango Bajo ===")
print(f"Rango de A: {np.linalg.matrix_rank(A)}")
print(f"Con k=1 (rango 1):")
U1, s1, Vt1 = svd_desde_cero(A, k=1)
A_rank1 = U1 @ np.diag(s1) @ Vt1
print(f"  Error relativo: {np.linalg.norm(A - A_rank1) / np.linalg.norm(A):.4f}")
print(f"  Varianza capturada: {s1[0]**2 / np.sum(sigma**2) * 100:.1f}%")

In [ ]:
# ============================================================
# SVD para compresión de imagen (aplicación práctica)
# ============================================================

# Crear una imagen sintética con estructura
np.random.seed(42)
size = 100
img = np.zeros((size, size))
# Gradientes y formas
for i in range(size):
    for j in range(size):
        img[i, j] = np.sin(i/10) * np.cos(j/10) + 0.3 * np.sin(i/5)
        if (i-50)**2 + (j-50)**2 < 400:
            img[i, j] += 1
img += np.random.randn(size, size) * 0.1

# SVD completa
U, s, Vt = np.linalg.svd(img, full_matrices=False)

# Compresión con diferentes rangos
rangos = [1, 5, 10, 20, 50]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Original
axes[0, 0].imshow(img, cmap='viridis')
axes[0, 0].set_title(f'Original (rango {np.linalg.matrix_rank(img)})', fontweight='bold')
axes[0, 0].axis('off')

for ax, k in zip(axes.flat[1:], rangos):
    img_k = U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]
    error = np.linalg.norm(img - img_k) / np.linalg.norm(img) * 100
    compresion = k * (size + size + 1) / (size * size) * 100
    ax.imshow(img_k, cmap='viridis')
    ax.set_title(f'Rango {k}\nError: {error:.1f}%, Datos: {compresion:.0f}%', fontweight='bold')
    ax.axis('off')

plt.suptitle('Compresión con SVD Truncada — Aplicación: Reducción de Dimensionalidad', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()

# Gráfico de valores singulares
fig2, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.semilogy(s, 'b.-')
ax1.set_xlabel('Índice'); ax1.set_ylabel('Valor singular (log)')
ax1.set_title('Espectro de Valores Singulares', fontweight='bold')
ax1.grid(True, alpha=0.3)

varianza_acum = np.cumsum(s**2) / np.sum(s**2) * 100
ax2.plot(varianza_acum, 'r-', lw=2)
ax2.axhline(95, color='k', linestyle='--', alpha=0.5, label='95%')
ax2.axhline(99, color='gray', linestyle='--', alpha=0.5, label='99%')
ax2.set_xlabel('Número de componentes'); ax2.set_ylabel('% Varianza capturada')
ax2.set_title('Varianza Acumulada', fontweight='bold')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()
print("✅ SVD truncada: mantener los k valores singulares más grandes = mejor aproximación de rango k")

In [ ]:
# ============================================================
# Otras descomposiciones: Cholesky, QR, LU
# ============================================================

def cholesky_desde_cero(A):
    """Descomposición de Cholesky: A = LL^T.
    Requiere: A simétrica y definida positiva.
    """
    n = A.shape[0]
    L = np.zeros_like(A)
    
    for i in range(n):
        for j in range(i + 1):
            suma = sum(L[i, k] * L[j, k] for k in range(j))
            if i == j:
                L[i, j] = np.sqrt(A[i, i] - suma)
            else:
                L[i, j] = (A[i, j] - suma) / L[j, j]
    return L

def qr_gram_schmidt(A):
    """Descomposición QR por Gram-Schmidt: A = QR.
    Q ortonormal, R triangular superior.
    """
    m, n = A.shape
    Q = np.zeros((m, n))
    R = np.zeros((n, n))
    
    for j in range(n):
        v = A[:, j].copy()
        for i in range(j):
            R[i, j] = np.dot(Q[:, i], A[:, j])
            v -= R[i, j] * Q[:, i]
        R[j, j] = np.linalg.norm(v)
        Q[:, j] = v / R[j, j]
    
    return Q, R

# --- Cholesky ---
print("=== Descomposición de Cholesky ===")
# Crear matriz definida positiva
B = np.random.randn(3, 3)
A_pd = B.T @ B + 3 * np.eye(3)  # Garantiza definida positiva

L = cholesky_desde_cero(A_pd)
print(f"A (def. positiva):\n{A_pd.round(4)}")
print(f"\nL (triangular inferior):\n{L.round(4)}")
print(f"\nLL^T =\n{(L @ L.T).round(4)}")
print(f"||A - LL^T|| = {np.linalg.norm(A_pd - L @ L.T):.1e}")
print("→ Uso: Gaussian Processes, sampling de normales multivariadas")

# --- QR ---
print("\n=== Descomposición QR ===")
A_qr = np.array([[1, 1, 0], [1, 0, 1], [0, 1, 1]], dtype=float)
Q, R = qr_gram_schmidt(A_qr)
print(f"Q (ortonormal):\n{Q.round(4)}")
print(f"\nR (triangular superior):\n{R.round(4)}")
print(f"\nQ^T Q (debe ser I):\n{(Q.T @ Q).round(4)}")
print(f"||A - QR|| = {np.linalg.norm(A_qr - Q @ R):.1e}")
print("→ Uso: Estabilidad numérica en mínimos cuadrados, QR iteration para eigenvalues")

# --- Comparar velocidad de resolver sistemas ---
print("\n=== Velocidad: resolver Ax = b ===")
import time
n = 500
A_test = np.random.randn(n, n)
A_test = A_test.T @ A_test + n * np.eye(n)
b_test = np.random.randn(n)

t0 = time.time()
x1 = np.linalg.solve(A_test, b_test)
t_solve = time.time() - t0

t0 = time.time()
L_test = np.linalg.cholesky(A_test)
y = np.linalg.solve(L_test, b_test)
x2 = np.linalg.solve(L_test.T, y)
t_cholesky = time.time() - t0

print(f"np.linalg.solve: {t_solve*1000:.2f} ms")
print(f"Cholesky solve:  {t_cholesky*1000:.2f} ms")
print(f"→ Cholesky es ~2x más rápido para matrices def. positivas")

## 7. Matrices Especiales en ML/DL

### Tipos importantes

| Tipo | Propiedad | Ejemplo en ML |
|------|-----------|---------------|
| **Simétrica** | $A = A^T$ | Covarianza, kernel, Hessiano |
| **Ortogonal** | $Q^TQ = I$, $Q^{-1} = Q^T$ | Rotaciones, SVD, inicialización |
| **Def. positiva** | $x^TAx > 0, \forall x \neq 0$ | Covarianza, kernels válidos |
| **Diagonal** | $a_{ij} = 0$ si $i \neq j$ | Escalado, $\Lambda$ en eigen |
| **Sparse** | Mayoría de elementos son 0 | Bag-of-words, grafos, embeddings |
| **Triangular** | Superior o inferior | Cholesky, atención causal |
| **Toeplitz** | Diagonales constantes | Convolución 1D |
| **Block diagonal** | Bloques independientes | Multi-head attention |

### Definida positiva — La más importante para ML

Una matriz simétrica $A$ es **definida positiva** si y solo si:
1. Todos sus eigenvalues son positivos: $\lambda_i > 0$
2. $x^TAx > 0$ para todo $x \neq 0$
3. Tiene descomposición de Cholesky
4. Todos los menores principales son positivos

La **matriz de covarianza** siempre es semidefinida positiva, lo que garantiza varianzas no negativas.

In [ ]:
# ============================================================
# Matrices especiales: propiedades y verificación
# ============================================================

def es_simetrica(A, tol=1e-10):
    return np.allclose(A, A.T, atol=tol)

def es_ortogonal(Q, tol=1e-10):
    n = Q.shape[0]
    return np.allclose(Q.T @ Q, np.eye(n), atol=tol)

def es_definida_positiva(A, tol=1e-10):
    if not es_simetrica(A):
        return False
    eigenvalues = np.linalg.eigvalsh(A)
    return np.all(eigenvalues > -tol)

def sparsity(A, tol=1e-10):
    return np.sum(np.abs(A) < tol) / A.size

# --- Demostración ---
print("=== Verificación de Matrices Especiales ===\n")

# Simétrica (covarianza)
datos = np.random.randn(100, 4)
cov = np.cov(datos.T)
print(f"Matriz de Covarianza (4x4):")
print(f"  Simétrica: {es_simetrica(cov)}")
print(f"  Def. positiva: {es_definida_positiva(cov)}")
print(f"  Eigenvalues: {np.linalg.eigvalsh(cov).round(4)}")

# Ortogonal (rotación)
theta = np.pi / 4
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta), np.cos(theta)]])
print(f"\nMatriz de Rotación 45°:")
print(f"  Ortogonal: {es_ortogonal(R)}")
print(f"  det(R) = {np.linalg.det(R):.6f} (1 = rotación, -1 = reflexión)")
print(f"  R^(-1) = R^T: {np.allclose(np.linalg.inv(R), R.T)}")

# Sparse (bag-of-words)
print(f"\nMatriz Sparse (simulando bag-of-words):")
vocab_size, n_docs = 1000, 50
bow = np.zeros((n_docs, vocab_size))
for i in range(n_docs):
    words = np.random.choice(vocab_size, size=20, replace=False)
    bow[i, words] = np.random.randint(1, 5, size=20)
print(f"  Shape: {bow.shape}")
print(f"  Sparsity: {sparsity(bow)*100:.1f}% ceros")
print(f"  → En la práctica usamos scipy.sparse para eficiencia")

# Máscara causal (Transformers)
print(f"\nMáscara Causal (Transformers):")
seq_len = 6
mask = np.tril(np.ones((seq_len, seq_len)))
print(mask.astype(int))
print("→ Triángular inferior: cada token solo atiende a tokens previos (GPT)")

## 8. Tensores — Generalizando a N dimensiones

### ¿Qué es un tensor?
Un tensor es la **generalización** de vectores y matrices a dimensiones arbitrarias:

| Dimensiones | Nombre | Ejemplo en DL |
|-------------|--------|--------------|
| 0D (escalar) | Tensor orden 0 | Loss value |
| 1D (vector) | Tensor orden 1 | Bias, embedding de una palabra |
| 2D (matriz) | Tensor orden 2 | Batch de features, pesos de capa |
| 3D | Tensor orden 3 | Secuencia de embeddings, imagen grayscale |
| 4D | Tensor orden 4 | Batch de imágenes: (batch, canales, alto, ancho) |
| 5D | Tensor orden 5 | Batch de videos: (batch, frames, canales, H, W) |

### Operaciones con tensores en DL

| Operación | Descripción | Ejemplo |
|-----------|-------------|---------|
| **Reshape** | Cambiar forma sin cambiar datos | (batch, seq, dim) → (batch, seq×dim) |
| **Transpose** | Permutar ejes | (batch, seq, dim) → (batch, dim, seq) |
| **Broadcasting** | Expandir dimensiones automáticamente | (batch, n, 1) + (1, 1, m) |
| **Einsum** | Notación Einstein para contracciones | Atención, outer product |
| **Batch matmul** | Multiplicación por batch | (B, N, K) @ (B, K, M) → (B, N, M) |

In [ ]:
# ============================================================
# Tensores: operaciones esenciales para Deep Learning
# ============================================================

print("=== Operaciones con Tensores (NumPy) ===\n")

# --- Shapes en DL ---
batch_size = 32
seq_len = 50
embed_dim = 128
num_heads = 8
head_dim = embed_dim // num_heads

# Simular embeddings de un Transformer
X = np.random.randn(batch_size, seq_len, embed_dim)
print(f"Input embeddings: {X.shape}  (batch, seq_len, embed_dim)")

# --- Reshape para Multi-Head Attention ---
# (batch, seq, embed) → (batch, heads, seq, head_dim)
X_multi = X.reshape(batch_size, seq_len, num_heads, head_dim)
X_multi = X_multi.transpose(0, 2, 1, 3)
print(f"Multi-head reshape: {X_multi.shape}  (batch, heads, seq, head_dim)")

# --- Batch Matrix Multiplication ---
# Attention scores: Q @ K^T / sqrt(d_k)
Q = np.random.randn(batch_size, num_heads, seq_len, head_dim)
K = np.random.randn(batch_size, num_heads, seq_len, head_dim)
V = np.random.randn(batch_size, num_heads, seq_len, head_dim)

# Esto multiplica matrices por batch: (B, H, S, D) @ (B, H, D, S) → (B, H, S, S)
scores = np.matmul(Q, K.transpose(0, 1, 3, 2)) / np.sqrt(head_dim)
print(f"Attention scores: {scores.shape}  (batch, heads, seq, seq)")

# Softmax por filas
def softmax(x, axis=-1):
    exp_x = np.exp(x - x.max(axis=axis, keepdims=True))
    return exp_x / exp_x.sum(axis=axis, keepdims=True)

attn_weights = softmax(scores)
print(f"Attention weights: {attn_weights.shape}")

# Weighted values
output = np.matmul(attn_weights, V)  # (B, H, S, S) @ (B, H, S, D) → (B, H, S, D)
print(f"Attention output: {output.shape}")

# --- Einsum: Notación Einstein ---
print("\n=== Einstein Summation (einsum) ===")

# Producto punto de vectores
a = np.random.randn(5)
b = np.random.randn(5)
print(f"Dot product: {np.einsum('i,i->', a, b):.4f} = {np.dot(a, b):.4f}")

# Multiplicación de matrices
A = np.random.randn(3, 4)
B = np.random.randn(4, 5)
C_einsum = np.einsum('ik,kj->ij', A, B)
print(f"Matmul: coincide = {np.allclose(C_einsum, A @ B)}")

# Batch matmul con einsum
Q_batch = np.random.randn(2, 3, 4)
K_batch = np.random.randn(2, 3, 4)
scores_einsum = np.einsum('bhsd,bhtd->bhst', 
                          Q_batch.reshape(2,1,3,4), K_batch.reshape(2,1,3,4))
print(f"Batch attention einsum: {scores_einsum.shape}")

# Traza
M = np.random.randn(4, 4)
print(f"Traza: {np.einsum('ii->', M):.4f} = {np.trace(M):.4f}")

# Outer product
u = np.array([1, 2, 3])
v = np.array([4, 5])
print(f"Outer product:\n{np.einsum('i,j->ij', u, v)}")

# --- Broadcasting ---
print("\n=== Broadcasting ===")
# Agregar bias a cada ejemplo del batch
batch = np.random.randn(32, 128)  # (batch, features)
bias = np.random.randn(128)       # (features,)
resultado = batch + bias           # Broadcasting: (32, 128) + (128,) → (32, 128)
print(f"Batch + bias: {batch.shape} + {bias.shape} → {resultado.shape}")

# Normalización por feature
mean = batch.mean(axis=0, keepdims=True)  # (1, 128)
std = batch.std(axis=0, keepdims=True)    # (1, 128)
normalized = (batch - mean) / (std + 1e-8)
print(f"Normalización: media={normalized.mean():.6f}, std={normalized.std():.4f}")

## 9. Aplicaciones Directas en ML/DL

### Gradientes como vectores
El gradiente de una función escalar respecto a un vector es un **vector** de derivadas parciales:

$$\nabla_w L = \begin{bmatrix} \frac{\partial L}{\partial w_1} \\ \frac{\partial L}{\partial w_2} \\ \vdots \\ \frac{\partial L}{\partial w_n} \end{bmatrix}$$

La actualización de pesos en gradient descent es:
$$w_{t+1} = w_t - \eta \nabla_w L$$

### Hessiano como matriz
La matriz Hessiana contiene las **segundas derivadas**:
$$H_{ij} = \frac{\partial^2 L}{\partial w_i \partial w_j}$$

- Si todos los eigenvalues de $H > 0$: **mínimo local**
- Si algún eigenvalue $< 0$: **punto de silla** (saddle point)
- En DL, la mayoría de puntos críticos son saddle points, no mínimos locales

### La capa Linear de un neural network
Una capa completamente conectada es una **transformación afín**:
$$h = \sigma(Wx + b)$$

Es literalmente una multiplicación matricial + bias + no-linealidad.

In [ ]:
# ============================================================
# Gradientes, Hessiano y Gradient Descent visualizado
# ============================================================

# Función cuadrática: f(x,y) = x² + 4y² (elipsoide)
def f(x, y):
    return x**2 + 4*y**2

def grad_f(x, y):
    return np.array([2*x, 8*y])

def hessian_f():
    return np.array([[2, 0], [0, 8]])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Superficie con gradientes
ax = axes[0]
x_grid = np.linspace(-4, 4, 50)
y_grid = np.linspace(-4, 4, 50)
X, Y = np.meshgrid(x_grid, y_grid)
Z = f(X, Y)

ax.contour(X, Y, Z, levels=20, cmap='viridis', alpha=0.7)
# Gradientes en puntos seleccionados
puntos = [(-3, 2), (2, -1), (-1, -2), (3, 1)]
for px, py in puntos:
    g = grad_f(px, py)
    g_norm = g / np.linalg.norm(g) * 0.8
    ax.annotate('', xy=(px - g_norm[0], py - g_norm[1]), xytext=(px, py),
                arrowprops=dict(arrowstyle='->', color='red', lw=2))
ax.set_title('Gradientes (dirección de -∇f)', fontweight='bold')
ax.set_xlabel('x'); ax.set_ylabel('y')

# 2. Gradient Descent
ax = axes[1]
ax.contour(X, Y, Z, levels=20, cmap='viridis', alpha=0.7)

# GD con diferentes learning rates
for lr, color, label in [(0.01, 'red', 'η=0.01'), (0.1, 'blue', 'η=0.1'), (0.24, 'green', 'η=0.24')]:
    w = np.array([3.5, 3.0])
    trayectoria = [w.copy()]
    for _ in range(30):
        g = grad_f(w[0], w[1])
        w = w - lr * g
        trayectoria.append(w.copy())
    trayectoria = np.array(trayectoria)
    ax.plot(trayectoria[:, 0], trayectoria[:, 1], f'{color[0]}o-', 
            markersize=3, lw=1.5, label=label, alpha=0.7)

ax.set_title('Gradient Descent: efecto del learning rate', fontweight='bold')
ax.legend(); ax.set_xlabel('x'); ax.set_ylabel('y')

# 3. Eigenvalues del Hessiano → curvatura
ax = axes[2]
H = hessian_f()
eigenvals, eigenvecs = np.linalg.eigh(H)

ax.contour(X, Y, Z, levels=20, cmap='viridis', alpha=0.7)
for i, (val, vec) in enumerate(zip(eigenvals, eigenvecs.T)):
    scale = val / eigenvals.max() * 2
    ax.annotate('', xy=vec*scale, xytext=[0,0],
                arrowprops=dict(arrowstyle='->', color='red', lw=3))
    ax.text(vec[0]*scale + 0.2, vec[1]*scale + 0.2, 
            f'λ={val:.0f}', color='red', fontsize=12, fontweight='bold')

ax.set_title('Eigenvalues del Hessiano = Curvatura', fontweight='bold')
ax.text(-3.5, -3, f'Condition number: {eigenvals.max()/eigenvals.min():.0f}\n'
        f'(más alto = más difícil optimizar)', fontsize=9, 
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
ax.set_xlabel('x'); ax.set_ylabel('y')

plt.tight_layout()
plt.savefig('gradientes_hessiano.png', dpi=100, bbox_inches='tight')
plt.show()
print("✅ El condition number del Hessiano determina cuán difícil es optimizar")
print(f"   Learning rate máximo estable ≈ 2/λ_max = {2/eigenvals.max():.2f}")

In [ ]:
# ============================================================
# Neural Network = Secuencia de transformaciones lineales
# ============================================================

def relu(x):
    return np.maximum(0, x)

def softmax_vec(x):
    exp_x = np.exp(x - x.max())
    return exp_x / exp_x.sum()

class RedNeuronal:
    """MLP simple para demostrar que es puro álgebra lineal."""
    
    def __init__(self, dims):
        """dims = [input, hidden1, hidden2, ..., output]"""
        self.pesos = []
        self.biases = []
        for i in range(len(dims) - 1):
            # Inicialización Xavier/Glorot
            std = np.sqrt(2.0 / (dims[i] + dims[i+1]))
            W = np.random.randn(dims[i], dims[i+1]) * std
            b = np.zeros(dims[i+1])
            self.pesos.append(W)
            self.biases.append(b)
    
    def forward(self, x, verbose=False):
        """Forward pass: secuencia de Wx + b + activación."""
        h = x
        if verbose:
            print(f"Input: {h.shape}")
        
        for i, (W, b) in enumerate(zip(self.pesos, self.biases)):
            # Transformación lineal: z = Wh + b
            z = h @ W + b
            if verbose:
                print(f"Capa {i+1}: {h.shape} @ {W.shape} + {b.shape} = {z.shape}")
            
            # Activación (ReLU excepto última capa)
            if i < len(self.pesos) - 1:
                h = relu(z)
            else:
                h = softmax_vec(z) if z.ndim == 1 else np.array([softmax_vec(zi) for zi in z])
        
        return h

# --- Demo: Clasificación ---
print("=== Red Neuronal como Álgebra Lineal ===\n")
print("Arquitectura: 784 → 256 → 128 → 10")
print("(Imagen MNIST → 2 capas ocultas → 10 dígitos)\n")

red = RedNeuronal([784, 256, 128, 10])
x_ejemplo = np.random.randn(784)  # "Imagen" aplanada

print("Forward pass (una imagen):")
output = red.forward(x_ejemplo, verbose=True)
print(f"\nProbabilidades de salida: {output.round(4)}")
print(f"Clase predicha: {output.argmax()}")

# Batch de imágenes
print(f"\nBatch forward (32 imágenes):")
batch = np.random.randn(32, 784)
output_batch = red.forward(batch, verbose=True)
print(f"Salida batch: {output_batch.shape}")

# Contar parámetros
total_params = sum(W.size + b.size for W, b in zip(red.pesos, red.biases))
print(f"\nTotal parámetros: {total_params:,}")
print("→ Cada parámetro es un número en una matriz o vector")
print("→ Entrenar = encontrar los valores óptimos via gradient descent")

In [ ]:
# ============================================================
# Inicialización de pesos: por qué importa el álgebra lineal
# ============================================================

def simular_forward(dims, init_fn, n_muestras=1000):
    """Simula un forward pass y mide la magnitud de activaciones."""
    activaciones_medias = []
    x = np.random.randn(n_muestras, dims[0])
    
    for i in range(len(dims) - 1):
        W = init_fn(dims[i], dims[i+1])
        x = x @ W
        x = np.maximum(0, x)  # ReLU
        activaciones_medias.append(np.mean(np.abs(x)))
    
    return activaciones_medias

dims = [256] * 20  # 20 capas de 256 neuronas

# Diferentes inicializaciones
inits = {
    'Muy pequeña (0.01)': lambda n_in, n_out: np.random.randn(n_in, n_out) * 0.01,
    'Muy grande (1.0)': lambda n_in, n_out: np.random.randn(n_in, n_out) * 1.0,
    'Xavier (√(1/n))': lambda n_in, n_out: np.random.randn(n_in, n_out) * np.sqrt(1.0 / n_in),
    'He (√(2/n))': lambda n_in, n_out: np.random.randn(n_in, n_out) * np.sqrt(2.0 / n_in),
    'Ortogonal': lambda n_in, n_out: np.linalg.qr(np.random.randn(n_in, n_out))[0][:, :n_out] if n_in >= n_out else np.linalg.qr(np.random.randn(n_out, n_in))[0][:n_in, :].T,
}

fig, ax = plt.subplots(figsize=(12, 6))
colores = ['red', 'orange', 'blue', 'green', 'purple']

for (nombre, init_fn), color in zip(inits.items(), colores):
    np.random.seed(42)
    activaciones = simular_forward(dims, init_fn)
    ax.semilogy(activaciones, 'o-', color=color, label=nombre, markersize=4)

ax.set_xlabel('Capa', fontsize=12)
ax.set_ylabel('Magnitud media de activaciones (log)', fontsize=12)
ax.set_title('Inicialización de Pesos: Vanishing vs Exploding Activations', fontweight='bold', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.axhline(1, color='k', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.savefig('inicializacion_pesos.png', dpi=100, bbox_inches='tight')
plt.show()
print("✅ He initialization mantiene la magnitud estable con ReLU")
print("   Xavier es mejor para tanh/sigmoid")
print("   Ortogonal preserva normas (eigenvalues = 1)")

In [ ]:
# ============================================================
# Matriz de Covarianza: el corazón del análisis multivariado
# ============================================================

def covarianza_desde_cero(X):
    """Calcula la matriz de covarianza: Σ = (1/(n-1)) (X-μ)^T (X-μ)"""
    n = X.shape[0]
    media = X.mean(axis=0)
    X_centrado = X - media
    return (X_centrado.T @ X_centrado) / (n - 1)

def correlacion_desde_cero(X):
    """Matriz de correlación: normalizar covarianza por desviaciones estándar."""
    cov = covarianza_desde_cero(X)
    stds = np.sqrt(np.diag(cov))
    return cov / np.outer(stds, stds)

# Dataset sintético: features con diferentes correlaciones
np.random.seed(42)
n = 500
x1 = np.random.randn(n)
x2 = 0.8 * x1 + 0.2 * np.random.randn(n)        # Alta correlación con x1
x3 = -0.5 * x1 + 0.5 * np.random.randn(n)         # Correlación negativa
x4 = np.random.randn(n)                              # Independiente
x5 = x2 + 0.1 * np.random.randn(n)                   # Casi igual a x2

X = np.column_stack([x1, x2, x3, x4, x5])
features = ['x1', 'x2 (≈0.8·x1)', 'x3 (≈-0.5·x1)', 'x4 (indep)', 'x5 (≈x2)']

# Calcular
cov_manual = covarianza_desde_cero(X)
corr_manual = correlacion_desde_cero(X)

print("=== Matriz de Covarianza (desde cero) ===")
print(cov_manual.round(3))
print(f"\nVerificación con NumPy: {np.allclose(cov_manual, np.cov(X.T))}")

# Visualizar
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Correlación
ax = axes[0]
im = ax.imshow(corr_manual, cmap='RdBu_r', vmin=-1, vmax=1)
for i in range(5):
    for j in range(5):
        ax.text(j, i, f'{corr_manual[i,j]:.2f}', ha='center', va='center', fontsize=10)
ax.set_xticks(range(5)); ax.set_yticks(range(5))
ax.set_xticklabels(features, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(features, fontsize=8)
ax.set_title('Matriz de Correlación', fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.8)

# Eigenvalues de la covarianza (varianza explicada)
ax = axes[1]
eigenvalues = np.linalg.eigvalsh(cov_manual)[::-1]
varianza_pct = eigenvalues / eigenvalues.sum() * 100
ax.bar(range(1, 6), varianza_pct, color='steelblue', alpha=0.7)
ax.plot(range(1, 6), np.cumsum(varianza_pct), 'ro-', lw=2)
ax.axhline(95, color='k', linestyle='--', alpha=0.5)
ax.set_xlabel('Componente Principal')
ax.set_ylabel('% Varianza')
ax.set_title('Eigenvalues de Covarianza = PCA', fontweight='bold')

# Scatter matrix
ax = axes[2]
eigenvalues_cov, eigenvectors_cov = np.linalg.eigh(cov_manual[:2, :2])
idx = eigenvalues_cov.argsort()[::-1]
eigenvalues_cov = eigenvalues_cov[idx]
eigenvectors_cov = eigenvectors_cov[:, idx]

ax.scatter(X[:, 0], X[:, 1], alpha=0.3, s=10)
for i in range(2):
    vec = eigenvectors_cov[:, i] * np.sqrt(eigenvalues_cov[i]) * 2
    ax.annotate('', xy=vec, xytext=[0,0],
                arrowprops=dict(arrowstyle='->', color='red', lw=3))
    ax.text(vec[0]+0.1, vec[1]+0.1, f'PC{i+1}', color='red', fontsize=12, fontweight='bold')
ax.set_xlabel('x1'); ax.set_ylabel('x2')
ax.set_title('Eigenvectors de Covarianza', fontweight='bold')
ax.set_aspect('equal'); ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()
print("✅ La matriz de covarianza codifica TODA la estructura lineal de los datos")
print("   Sus eigenvectors = PCA, sus eigenvalues = varianza por componente")

## 10. Resumen: Álgebra Lineal en ML/DL

### Mapa conceptual

```
ÁLGEBRA LINEAL EN ML/DL
│
├── VECTORES
│   ├── Features de un ejemplo
│   ├── Pesos del modelo
│   ├── Gradientes (∇L)
│   ├── Embeddings (word2vec, BERT)
│   └── Operaciones: dot product → neurona, coseno → similitud
│
├── MATRICES
│   ├── Dataset X (n_samples × n_features)
│   ├── Pesos W de cada capa
│   ├── Covarianza → PCA, Gaussian Processes
│   ├── Atención (seq × seq) en Transformers
│   └── Operaciones: A@B → forward pass, A⁻¹ → ecuación normal
│
├── DESCOMPOSICIONES
│   ├── Eigen → PCA, espectral clustering, estabilidad
│   ├── SVD → reducción dimensión, NLP (LSA), compresión
│   ├── Cholesky → Gaussian Processes, sampling
│   └── QR → estabilidad numérica
│
├── TENSORES
│   ├── Batches (4D para imágenes, 3D para secuencias)
│   ├── Multi-head attention (reshape + transpose)
│   ├── Broadcasting (bias, normalización)
│   └── Einsum (contracciones generales)
│
└── OPTIMIZACIÓN
    ├── Gradiente → dirección de descenso
    ├── Hessiano → curvatura, saddle points
    ├── Condition number → dificultad de optimizar
    └── Inicialización → Xavier, He, Ortogonal
```

### Fórmulas esenciales para recordar

| Fórmula | Significado |
|---------|-------------|
| $y = Wx + b$ | Capa neural = transformación afín |
| $w^* = (X^TX)^{-1}X^Ty$ | Regresión lineal (ecuación normal) |
| $A = V\Lambda V^{-1}$ | Eigendecomposition (PCA) |
| $A = U\Sigma V^T$ | SVD (reducción dimensión) |
| $\text{Attn}(Q,K,V) = \text{softmax}(QK^T/\sqrt{d_k})V$ | Self-Attention |
| $w_{t+1} = w_t - \eta \nabla L$ | Gradient descent |
| $\cos(\theta) = \frac{u \cdot v}{\|u\|\|v\|}$ | Similitud coseno |

In [ ]:
# ============================================================
# EJERCICIO INTEGRADOR: PCA desde cero usando álgebra lineal
# ============================================================

class PCA_AlgebraLineal:
    """PCA implementado paso a paso con álgebra lineal pura."""
    
    def __init__(self, n_componentes):
        self.n_componentes = n_componentes
        self.componentes = None
        self.varianza_explicada = None
        self.media = None
    
    def fit(self, X):
        # Paso 1: Centrar datos (restar media → vector)
        self.media = X.mean(axis=0)
        X_centrado = X - self.media
        
        # Paso 2: Matriz de covarianza (multiplicación matricial)
        n = X.shape[0]
        covarianza = (X_centrado.T @ X_centrado) / (n - 1)
        
        # Paso 3: Eigendecomposition
        eigenvalues, eigenvectors = np.linalg.eigh(covarianza)
        
        # Paso 4: Ordenar por eigenvalue (mayor a menor)
        idx = eigenvalues.argsort()[::-1]
        eigenvalues = eigenvalues[idx]
        eigenvectors = eigenvectors[:, idx]
        
        # Paso 5: Seleccionar top-k componentes
        self.componentes = eigenvectors[:, :self.n_componentes]
        self.varianza_explicada = eigenvalues[:self.n_componentes] / eigenvalues.sum()
        
        return self
    
    def transform(self, X):
        # Proyección: multiplicación matricial X_c @ V_k
        X_centrado = X - self.media
        return X_centrado @ self.componentes
    
    def inverse_transform(self, X_reducido):
        # Reconstrucción: X_rec = X_red @ V_k^T + μ
        return X_reducido @ self.componentes.T + self.media

# --- Aplicar a datos sintéticos ---
np.random.seed(42)
n_muestras = 500

# Datos 3D con estructura 2D
t = np.random.uniform(0, 2*np.pi, n_muestras)
x = 3 * np.cos(t) + np.random.randn(n_muestras) * 0.2
y = 2 * np.sin(t) + np.random.randn(n_muestras) * 0.2
z = 0.5 * x + 0.3 * y + np.random.randn(n_muestras) * 0.1
X_3d = np.column_stack([x, y, z])

# PCA
pca = PCA_AlgebraLineal(n_componentes=2).fit(X_3d)
X_2d = pca.transform(X_3d)
X_rec = pca.inverse_transform(X_2d)

fig = plt.figure(figsize=(16, 5))

# 3D original
ax1 = fig.add_subplot(131, projection='3d')
ax1.scatter(X_3d[:, 0], X_3d[:, 1], X_3d[:, 2], c=t, cmap='hsv', s=10, alpha=0.6)
ax1.set_title('Datos originales 3D', fontweight='bold')
ax1.set_xlabel('x'); ax1.set_ylabel('y'); ax1.set_zlabel('z')

# 2D proyección
ax2 = fig.add_subplot(132)
ax2.scatter(X_2d[:, 0], X_2d[:, 1], c=t, cmap='hsv', s=10, alpha=0.6)
ax2.set_title(f'Proyección PCA 2D\nVar. explicada: {pca.varianza_explicada.sum()*100:.1f}%', fontweight='bold')
ax2.set_xlabel('PC1'); ax2.set_ylabel('PC2')
ax2.grid(True, alpha=0.3)

# 3D reconstruido
ax3 = fig.add_subplot(133, projection='3d')
ax3.scatter(X_rec[:, 0], X_rec[:, 1], X_rec[:, 2], c=t, cmap='hsv', s=10, alpha=0.6)
error = np.linalg.norm(X_3d - X_rec) / np.linalg.norm(X_3d) * 100
ax3.set_title(f'Reconstrucción desde 2D\nError: {error:.1f}%', fontweight='bold')
ax3.set_xlabel('x'); ax3.set_ylabel('y'); ax3.set_zlabel('z')

plt.tight_layout(); plt.show()

print("=== PCA = Álgebra Lineal Pura ===")
print(f"Varianza explicada: PC1={pca.varianza_explicada[0]*100:.1f}%, PC2={pca.varianza_explicada[1]*100:.1f}%")
print(f"Error de reconstrucción: {error:.2f}%")
print("\n📐 Todo se reduce a: centrar → covarianza → eigendecomposition → proyectar")